<a href="https://colab.research.google.com/github/Ahmed-Rafi07/-MongoDB-connection-string-in-the-.env-file-/blob/main/Naive_Bayes_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the dataset
df = pd.read_csv("patient_records.csv")

# Drop rows where the target label 'diagnosis' is missing
df = df.dropna(subset=["diagnosis"])

# Filter out classes that have only 1 member to allow stratification
class_counts = df["diagnosis"].value_counts()
valid_classes = class_counts[class_counts >= 2].index
df = df[df["diagnosis"].isin(valid_classes)]

# Setup columns that actually exist in your dataset:
num_cols = ["age", "cholesterol", "bmi"]

# Clean numeric columns: coerce non-numeric strings (like 'fifty') to NaN
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill or drop NaNs introduced by coercion
df = df.dropna(subset=num_cols)

# Since 'symptom_notes' is not in this dataset, we will use 'smoking_status' as our text/categorical feature
text_col = "smoking_status"

# One shared train-test split
train, test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["diagnosis"]
)

print("Dataset split successfully!")
print("Train size:", train.shape[0], "| Test size:", test.shape[0])

Dataset split successfully!
Train size: 699 | Test size: 175


### Block 1 — View A: GaussianNB on the Numeric Vitals
First view: the five continuous vitals. Continuous numbers → GaussianNB, which assumes each feature is normally distributed within each class.

In [13]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report
import pandas as pd

# Safely copy to avoid SettingWithCopyWarning
train_clean = train.copy()
test_clean = test.copy()

# Convert all numeric columns to float, turning string values like 'fifty' into NaN
for col in num_cols:
    train_clean[col] = pd.to_numeric(train_clean[col], errors='coerce')
    test_clean[col] = pd.to_numeric(test_clean[col], errors='coerce')

# Drop rows where any numerical feature is NaN
train_clean = train_clean.dropna(subset=num_cols)
test_clean = test_clean.dropna(subset=num_cols)

# Train GaussianNB on fully cleaned continuous numerical columns
gnb = GaussianNB().fit(train_clean[num_cols], train_clean["diagnosis"])
print("=== GaussianNB Vitals Report ===")
print(classification_report(test_clean["diagnosis"], gnb.predict(test_clean[num_cols]), zero_division=0))

=== GaussianNB Vitals Report ===
               precision    recall  f1-score   support

       Cancer       0.00      0.00      0.00         8
     Diabetes       0.00      0.00      0.00        19
      Healthy       0.34      0.67      0.45        36
Heart Disease       0.00      0.00      0.00        21
 Hypertension       0.13      0.48      0.21        23
        Other       0.44      0.10      0.17        68

     accuracy                           0.24       175
    macro avg       0.15      0.21      0.14       175
 weighted avg       0.26      0.24      0.18       175



### Block 2 — View B: MultinomialNB on the Symptom Notes
Second view: the free-text `symptom_notes`. Since text consists of discrete word frequencies, we use `CountVectorizer` to generate token count matrices fit for `MultinomialNB`.

In [11]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# Train MultinomialNB on 'smoking_status' as the text feature representation
vec = CountVectorizer()
Xtr = vec.fit_transform(train[text_col].fillna("Unknown"))
Xte = vec.transform(test[text_col].fillna("Unknown"))

mnb = MultinomialNB(alpha=1.0).fit(Xtr, train["diagnosis"])
print("=== MultinomialNB Text/Category Report ===")
print(classification_report(test["diagnosis"], mnb.predict(Xte), zero_division=0))

=== MultinomialNB Text/Category Report ===
               precision    recall  f1-score   support

       Cancer       0.00      0.00      0.00         9
     Diabetes       0.00      0.00      0.00        21
      Healthy       0.00      0.00      0.00        40
Heart Disease       0.00      0.00      0.00        24
 Hypertension       0.00      0.00      0.00        25
        Other       0.39      1.00      0.56        75

     accuracy                           0.39       194
    macro avg       0.06      0.17      0.09       194
 weighted avg       0.15      0.39      0.22       194

